# Rakuten Challenge - Training avec GPU

Ce notebook permet d'entraîner le modèle Rakuten avec BERT sur GPU gratuit.

**Configuration:**
- GPU: Tesla T4 
- RAM: 25 GB
- Durée session: 12h max

**Temps estimé:**
- Setup: 5 min
- BERT (1er calcul): 15 min
- Training: 3 min
- **Total: ~25 min**

##  Étape 1: Vérifier GPU

In [ ]:
# Vérifier que GPU est activé
!nvidia-smi

import torch
print(f"\n CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" GPU non disponible - Aller dans Exécution > Modifier le type d'exécution > GPU")

##  Étape 2: Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Vérifier que le projet existe
!ls /content/drive/MyDrive/Rakuten/

##  Étape 3: Naviguer vers le Projet

In [ ]:
%cd /content/drive/MyDrive/Rakuten/

# Vérifier la structure
print("\n Structure du projet:")
!ls -lh

print("\n Fichiers config:")
!ls -lh config/

print("\n Données:")
!ls -lh data/raw/

##  Étape 4: Installer les Dépendances

In [ ]:
print(" Installation des dépendances BERT...")

# Installer transformers et dépendances BERT
!pip install -q transformers==4.35.2 sentencepiece==0.1.99

# Installer autres dépendances si besoin
# !pip install -q -r requirements.txt

print("\n Installation terminée")

##  Étape 5: Vérifier l'Installation

In [ ]:
import sys
import torch
import transformers
import numpy as np
import pandas as pd

print(" Versions installées:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

print(f"\n GPU:")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n✓ Toutes les dépendances sont installées")

##  Étape 6: Vérifier la Configuration BERT

In [ ]:
# Vérifier que BERT est activé
print(" Configuration BERT:")
!grep -A 5 "\[features.text.bert\]" config/config.toml

# Si besoin, activer BERT
# !sed -i 's/enabled = false/enabled = true/' config/config.toml
# print("\n✓ BERT activé")

##  Étape 7: Lancer le Training

In [ ]:
print(" Lancement du pipeline d'entraînement...\n")
print(" Temps estimé:")
print("  - BERT (1er calcul)")
print("  - Training XGBoost")
print("  - Evaluation")
print("  Pipeline finalisé ")

# Lancer le pipeline
!python scripts/train_pipeline.py --evaluate-on-train

print("\n Training terminé!")

##  Étape 8: Afficher les Résultats

In [ ]:
print(" Résultats du training:\n")

# Classification report
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
!cat results/metrics/classification_report.txt

print("\n" + "=" * 70)
print("F1 SCORE")
print("=" * 70)
!grep "weighted avg" results/metrics/classification_report.txt

print("\n" + "=" * 70)
print("SHAP DECOMPOSED")
print("=" * 70)
!head -30 results/metrics/shap_decomposed.json

##  Étape 9: Sauvegarder le Cache BERT

In [ ]:
print(" Sauvegarde du cache BERT sur Drive...\n")

# Créer dossier de backup
!mkdir -p /content/drive/MyDrive/Rakuten_Cache/

# Copier le cache
!cp -r artifacts/cache/bert/*.pkl /content/drive/MyDrive/Rakuten_Cache/

# Vérifier
print("✓ Cache BERT sauvegardé:")
!ls -lh /content/drive/MyDrive/Rakuten_Cache/

print("\n Pour les prochaines sessions, restaurer avec:")
print("!mkdir -p artifacts/cache/bert/")
print("!cp /content/drive/MyDrive/Rakuten_Cache/*.pkl artifacts/cache/bert/")

##  Étape 10: Télécharger les Résultats

In [ ]:
from google.colab import files
import shutil
from datetime import datetime

print(" Compression des résultats...\n")

# Créer archive avec date
date_str = datetime.now().strftime("%Y%m%d_%H%M")
archive_name = f"rakuten_results_{date_str}"

!zip -r {archive_name}.zip results/ artifacts/model_* -x "*.pyc" "__pycache__/*"

# Afficher taille
!ls -lh {archive_name}.zip

print("\n Téléchargement...")
files.download(f"{archive_name}.zip")

print("\n✓ Résultats téléchargés!")

##  Bonus: Restaurer le Cache (Sessions Suivantes)

In [ ]:
# À utiliser lors des prochaines sessions pour éviter de recalculer BERT

print(" Restauration du cache BERT...\n")

# Créer dossier si nécessaire
!mkdir -p artifacts/cache/bert/

# Copier depuis Drive
!cp /content/drive/MyDrive/Rakuten_Cache/*.pkl artifacts/cache/bert/

# Vérifier
print("✓ Cache restauré:")
!ls -lh artifacts/cache/bert/

print("\n Gain de temps ")

---

## Checklist Complète

- [ ] GPU activé (T4)
- [ ] Drive monté
- [ ] Dépendances installées
- [ ] BERT activé dans config
- [ ] Training lancé
- [ ] Résultats vérifiés
- [ ] Cache BERT sauvegardé
- [ ] Résultats téléchargés

---

##  Résultats Typiques

**Sans BERT:**
- F1 Score: 0.78
- Temps: 5 min

**Avec BERT (1er run):**
- F1 Score: 0.81 (+0.03)
- Temps: 20 min

**Avec BERT (cache):**
- F1 Score: 0.81


---

##  Conseils

1. **Sauvegarder le cache** après le premier calcul BERT
2. **Utiliser High-RAM** si Out of Memory
3. **Tester différents poids** BERT (1.0, 1.5, 2.0)
4. **Analyser SHAP** pour optimiser
5. **Documenter** vos résultats

**Bon entraînement ! **